## Agent Strategy Overview

This agent utilizes a **Factory-Centric Convoy Strategy** governed by strict action hierarchies, collision avoidance, and dynamic BFS pathfinding. 

#### 1. Execution Priority & Safety
* **Hierarchy:** Units process actions in a strict order: `FACTORY` > `WORKER` > `OTHERS`.
* **Collision Avoidance:** Every valid move or jump logs its destination into a global `reserved` set, ensuring lower-priority units never step on the toes of high-priority units.

#### 2. Factory Logic (The VIP)
* **Lava Panic Evasion:** Instantly executes `JUMP_NORTH` if the scrolling southern boundary closes within 2 tiles.
* **Dual-Layer Pathfinding:** Searches for the optimal path up to 20 tiles North using standard movement (`bfs_first_step`). If completely blocked, it falls back to a specialized jump-aware pathfinder (`bfs_jump_step`) to vault out of traps.
* **Umbilical Energy Feed:** Monitors adjacent Workers; if their energy dips $\le 150$, the Factory transfers its own energy to keep their drills powered.
* **Emergency Spawning:** If boxed in with no path/jump available and 0 Workers on the board, it spawns a Worker directly North to act as a drill.

#### 3. Worker Logic (The Vanguard)
* **Contextual Targeting:**
  * *High Threat (Boundary Gap $\le$ 4):* Abandons resource gathering and locks onto a coordinate strictly ahead of the Factory to carve an escape tunnel.
  * *Low Threat (Safe):* Hunts nearby crystals within a 10-tile radius to build energy reserves.
* **Snowplow Protocol:** Automatically triggers `REMOVE_NORTH` if touching a northern wall and holding $\ge 100$ energy.

In [ ]:
!pip install -qU kaggle-environments

### The Agent Function


In [ ]:
%%writefile submission.py
from collections import deque

FACTORY = 0
SCOUT = 1
WORKER = 2
MINER = 3

DIRS = ("NORTH", "EAST", "WEST", "SOUTH")
OFFSETS = {"NORTH": (0, 1), "EAST": (1, 0), "WEST": (-1, 0), "SOUTH": (0, -1)}
WALL_BITS = {"NORTH": 1, "EAST": 2, "SOUTH": 4, "WEST": 8}


def agent(obs, config):
    actions = {}
    width = config.width
    south = obs.southBound
    north = obs.northBound

    my_robots = {uid: d for uid, d in obs.robots.items() if d[4] == obs.player}
    occupied = {(d[1], d[2]): uid for uid, d in my_robots.items()}
    reserved = set()

    def get_w(c, r):
        idx = (r - south) * width + c
        if 0 <= c < width and 0 <= idx < len(obs.walls) and obs.walls[idx] != -1:
            return obs.walls[idx]
        return 0

    def can_move(c, r, d):
        dc, dr = OFFSETS[d]
        nc, nr = c + dc, r + dr
        if not (0 <= nc < width and south <= nr <= north):
            return False
        if get_w(c, r) & WALL_BITS[d]:
            return False
        return True

    def can_jump(c, r, d):
        dc, dr = OFFSETS[d]
        nc, nr = c + 2 * dc, r + 2 * dr
        if not (0 <= nc < width and south <= nr <= north):
            return False
        if get_w(nc, nr) == 15:
            return False
        return True

    def record_action(uid, act, res=None):
        actions[uid] = act
        if res:
            reserved.add(res)

    def bfs_first_step(start, goals, depth=20):
        if not goals:
            return None
        q = deque([(start, None, 0)])
        seen = {start}
        g_set = set(goals)
        while q:
            pos, first_d, dist = q.popleft()
            if pos in g_set and dist > 0:
                return first_d
            if dist >= depth:
                continue
            for d in DIRS:
                if can_move(pos[0], pos[1], d):
                    nxt = (pos[0] + OFFSETS[d][0], pos[1] + OFFSETS[d][1])
                    if nxt != start and nxt in occupied:
                        continue
                    if nxt not in seen:
                        seen.add(nxt)
                        q.append((nxt, first_d or d, dist + 1))
        return None

    def bfs_jump_step(start, goals, jump_cd, depth=20):
        if not goals:
            return None
        q = deque([(0, start, None, jump_cd)])
        best_j = {start: jump_cd}
        g_set = set(goals)
        while q:
            dist, pos, first_d, j_rem = q.popleft()
            if pos in g_set and dist > 0:
                return first_d
            if dist >= depth:
                continue

            for d in DIRS:
                if can_move(pos[0], pos[1], d):
                    nxt = (pos[0] + OFFSETS[d][0], pos[1] + OFFSETS[d][1])
                    if nxt != start and nxt in occupied:
                        continue
                    nj = max(0, j_rem - 1)
                    if nxt not in best_j or nj < best_j[nxt]:
                        best_j[nxt] = nj
                        q.append((dist + 1, nxt, first_d or d, nj))

            if j_rem == 0:
                for d in DIRS:
                    if can_jump(pos[0], pos[1], d):
                        nxt = (pos[0] + 2 * OFFSETS[d][0], pos[1] + 2 * OFFSETS[d][1])
                        if nxt != start and nxt in occupied:
                            continue
                        nj = 20
                        if nxt not in best_j or nj < best_j[nxt]:
                            best_j[nxt] = nj
                            q.append((dist + 1, nxt, first_d or f"JUMP_{d}", nj))
        return None

    counts = {rt: sum(1 for d in my_robots.values() if d[0] == rt) for rt in range(4)}
    crystals = {tuple(map(int, k.split(","))): v for k, v in obs.crystals.items()}

    ordered = sorted(
        my_robots.items(),
        key=lambda item: (
            (0 if item[1][0] == FACTORY else 1 if item[1][0] == WORKER else 2),
            item[0],
        ),
    )

    for uid, data in ordered:
        robot_type = data[0]
        col, row = data[1], data[2]
        energy = data[3]
        move_cd = data[5] if len(data) > 5 else 0
        jump_cd = data[6] if len(data) > 6 else 0
        build_cd = data[7] if len(data) > 7 else 0

        gap = row - south

        if robot_type == FACTORY:
            emergency = gap <= 2 and south > 0
            if emergency and jump_cd == 0 and can_jump(col, row, "NORTH"):
                nxt = (col, row + 2)
                if nxt not in reserved and nxt not in occupied:
                    record_action(uid, "JUMP_NORTH", nxt)
                    continue

            worker_d = next((d for d in my_robots.values() if d[0] == WORKER), None)
            if worker_d:
                wc, wr, wen = worker_d[1], worker_d[2], worker_d[3]
                if abs(col - wc) + abs(row - wr) == 1 and wen <= 150:
                    for d in DIRS:
                        if col + OFFSETS[d][0] == wc and row + OFFSETS[d][1] == wr:
                            record_action(uid, f"TRANSFER_{d}")
                            break
                    if uid in actions:
                        continue

            max_depth = 20
            step = bfs_first_step(
                (col, row),
                [(tc, min(north, row + max_depth)) for tc in range(width)],
                depth=max_depth,
            )

            if not step:
                if jump_cd <= 1:
                    step = bfs_jump_step(
                        (col, row),
                        [(tc, min(north, row + max_depth)) for tc in range(width)],
                        jump_cd,
                        depth=max_depth,
                    )

            if not step:
                spawn = (col, row + 1)
                if (
                    build_cd <= 1
                    and row + 1 <= north
                    and not (get_w(col, row) & 1)
                    and spawn not in occupied
                    and spawn not in reserved
                    and counts[WORKER] < 1
                    and energy >= config.workerCost
                ):
                    record_action(uid, "BUILD_WORKER", spawn)
                    counts[WORKER] += 1
                    continue

            if move_cd <= 1:
                if step:
                    if step.startswith("JUMP_"):
                        d = step.split("_")[1]
                        nxt = (col + 2 * OFFSETS[d][0], row + 2 * OFFSETS[d][1])
                    else:
                        nxt = (col + OFFSETS[step][0], row + OFFSETS[step][1])
                    if nxt not in reserved and nxt not in occupied:
                        record_action(uid, step, nxt)
                        continue

                record_action(uid, "IDLE")
            else:
                record_action(uid, "IDLE")

        elif robot_type == WORKER:
            f_uid = next((u for u, d in my_robots.items() if d[0] == FACTORY), None)
            if f_uid:
                fx, fy = my_robots[f_uid][1], my_robots[f_uid][2]

                if gap <= 4:
                    w_targets = [(fx, min(north, fy + 5))]
                else:
                    targets = [
                        c for c in crystals if abs(c[0] - col) + abs(c[1] - row) <= 10
                    ]
                    if targets:
                        w_targets = sorted(
                            targets, key=lambda t: abs(t[0] - col) + abs(t[1] - row)
                        )[:3]
                    else:
                        w_targets = [(fx, min(north, fy + 5))]

                if (get_w(col, row) & 1) and energy >= 100:
                    record_action(uid, "REMOVE_NORTH")
                    continue

                if move_cd <= 1:
                    w_step = bfs_first_step((col, row), w_targets, depth=8)
                    if w_step:
                        nxt = (col + OFFSETS[w_step][0], row + OFFSETS[w_step][1])
                        if nxt not in reserved and nxt not in occupied:
                            record_action(uid, w_step, nxt)
                            continue

                    # Fallback to force North
                    if can_move(col, row, "NORTH"):
                        nxt = (col, row + 1)
                        if nxt not in reserved and nxt not in occupied:
                            record_action(uid, "NORTH", nxt)
                            continue

                    record_action(uid, "IDLE")
                else:
                    record_action(uid, "IDLE")

        if uid not in actions:
            record_action(uid, "IDLE")

    return actions

### Play vs. Random Agent


In [ ]:
from kaggle_environments import make

env = make("crawl", debug=True)

# Run match: Our Agent vs Random
env.run(["submission.py", "random"])

for i, state in enumerate(env.steps[-1]):
    print(f"Player {i+1} Status: {state.status}, Reward: {state.reward}")

env.render(mode="ipython", width=800, height=600)